<a href="https://colab.research.google.com/github/mimomaina/Career-Path-Recommendation-System/blob/main/FAISS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
# !pip install faiss-cpu


In [12]:
# Load dataset with low_memory=False to avoid DtypeWarning
df = pd.read_csv("/content/tech_industry_dataset.csv", low_memory=False)

job_embeddings =np.load("/content/job_embeddings_full.npy")

# Verify shapes before proceeding
print(f"Dataset Shape: {df.shape}")
print(f"Embeddings Shape: {job_embeddings.shape}")

df = df.iloc[:job_embeddings.shape[0]].reset_index(drop=True)


Dataset Shape: (293130, 22)
Embeddings Shape: (292167, 384)


In [13]:
from sklearn.cluster import MiniBatchKMeans

# Define & Train MiniBatch K-Means Model
num_clusters = 30
kmeans = MiniBatchKMeans(n_clusters=num_clusters, batch_size=10000, random_state=42)

df["Cluster"] = kmeans.fit_predict(job_embeddings)

# Display Sample Clustered Jobs
print("Job clustering complete. Sample clusters:")
print(df.groupby("Cluster")["Job Title"].unique().head())

# Save the Clustered Dataset
df.to_csv("clustered_jobs.csv", index=False)

# Save Cluster Centroids for Future Use
np.save("cluster_centroids.npy", kmeans.cluster_centers_)


Job clustering complete. Sample clusters:
Cluster
0        [Data Engineer, Data Analyst, Data Scientist]
1    [Software Engineer, Database Developer, Custom...
2    [Software Tester, Back-End Developer, Quality ...
3    [Network Technician, Software Engineer, Databa...
4    [Software Engineer, Java Developer, Back-End D...
Name: Job Title, dtype: object


In [14]:
# Step 1: Normalize embeddings for cosine similarity
job_embeddings = job_embeddings / np.linalg.norm(job_embeddings, axis=1, keepdims=True)

# Step 2: Create a FAISS index for cosine similarity
index = faiss.IndexFlatIP(dimension)
index.add(job_embeddings)

# Step 3: Encode user input with weighted skills
weighted_skills = "Python + Python + Machine Learning + SQL"
user_embedding = model.encode(weighted_skills)
user_embedding = user_embedding.astype(np.float32).reshape(1, -1)
user_embedding = user_embedding / np.linalg.norm(user_embedding, axis=1, keepdims=True)

# Step 4: Perform similarity search
distances, indices = index.search(user_embedding, k=10)

# Step 5: Filter results for diversity
unique_titles = set()
diverse_recommendations = []
for idx in indices.flatten():
    title = df.iloc[idx]["Job Title"]
    if title not in unique_titles:
        diverse_recommendations.append(df.iloc[idx])
        unique_titles.add(title)
    if len(diverse_recommendations) == 5:  # Top 5 diverse results
        break

# Display diverse recommendations
print("Diverse Recommendations:")
for job in diverse_recommendations:
    print(job["Job Title"], job["Cluster"])

Diverse Recommendations:
Data Scientist 0


In [23]:
#Save the FAISS Index to Disk
faiss.write_index(index, "faiss_index.index")
print("FAISS index saved to 'faiss_index.index'.")

#Load the FAISS Index from Disk
loaded_index = faiss.read_index("faiss_index.index")
print(f"FAISS index loaded. Indexed {loaded_index.ntotal} job embeddings.")

FAISS index saved to 'faiss_index.index'.
FAISS index loaded. Indexed 292167 job embeddings.


In [16]:
print("Recommended Role:", job["Job Title"])
print("Cluster:", job["Cluster"])
print("Required Skills:", job["skills"])  # Assuming "Skills" column exists

Recommended Role: Data Scientist
Cluster: 0
Required Skills: Machine learning algorithms Python programming Data preprocessing Deep learning Model evaluation


In [18]:
required_skills = set(job["skills"].split(", "))
user_skills = set(["Python", "SQL"])
missing_skills = required_skills - user_skills
print(f"Missing Skills: {missing_skills}")

Missing Skills: {'Machine learning algorithms Python programming Data preprocessing Deep learning Model evaluation'}


In [19]:
# Define test cases (user inputs for different roles)
test_cases = {
    "Data Scientist": "Python, Machine Learning, SQL",
    "Software Engineer": "Python, Java, Software Development, Algorithms",
    "DevOps Engineer": "AWS, Kubernetes, CI/CD, Docker",
    "UX Designer": "User Experience, Wireframing, Prototyping, Figma",
    "Data Engineer": "SQL, ETL, Big Data, Apache Spark",
    "Product Manager": "Agile, Roadmapping, Stakeholder Management, Product Lifecycle"
}

# Loop through each test case and generate recommendations
for role, skills in test_cases.items():
    print(f"\nTesting for Role: {role}")
    print(f"Skills: {skills}")

    # Encode user input
    user_embedding = model.encode(skills)
    user_embedding = user_embedding.astype(np.float32).reshape(1, -1)
    user_embedding = user_embedding / np.linalg.norm(user_embedding, axis=1, keepdims=True)

    # Perform similarity search
    distances, indices = index.search(user_embedding, k=10)

    # Filter results for diversity
    unique_titles = set()
    diverse_recommendations = []
    for idx in indices.flatten():
        title = df.iloc[idx]["Job Title"]
        if title not in unique_titles:
            diverse_recommendations.append((title, df.iloc[idx]["Cluster"]))
            unique_titles.add(title)
        if len(diverse_recommendations) == 5:  # Top 5 diverse results
            break

    # Display results
    print("Diverse Recommendations:")
    for job_title, cluster in diverse_recommendations:
        print(f"- {job_title} (Cluster {cluster})")


Testing for Role: Data Scientist
Skills: Python, Machine Learning, SQL
Diverse Recommendations:
- Data Analyst (Cluster 0)

Testing for Role: Software Engineer
Skills: Python, Java, Software Development, Algorithms
Diverse Recommendations:
- Software Developer (Cluster 20)

Testing for Role: DevOps Engineer
Skills: AWS, Kubernetes, CI/CD, Docker
Diverse Recommendations:
- Systems Engineer (Cluster 6)

Testing for Role: UX Designer
Skills: User Experience, Wireframing, Prototyping, Figma
Diverse Recommendations:
- Front-End Developer (Cluster 20)

Testing for Role: Data Engineer
Skills: SQL, ETL, Big Data, Apache Spark
Diverse Recommendations:
- Data Engineer (Cluster 0)

Testing for Role: Product Manager
Skills: Agile, Roadmapping, Stakeholder Management, Product Lifecycle
Diverse Recommendations:
- Network Administrator (Cluster 6)


In [22]:
# Define test cases (user inputs for different roles)
test_cases = {
    "Data Scientist": "Python, Machine Learning, SQL",
    "Software Engineer": "Python, Java, Software Development, Algorithms",
    "DevOps Engineer": "AWS, Kubernetes, CI/CD, Docker",
    "UX Designer": "User Experience, Wireframing, Prototyping, Figma",
    "Data Engineer": "SQL, ETL, Big Data, Apache Spark",
    "Product Manager": "Agile, Roadmapping, Stakeholder Management, Product Lifecycle"
}

# Loop through each test case and generate recommendations
for role, skills in test_cases.items():
    print(f"\nTesting for Role: {role}")
    print(f"Skills: {skills}")

    # Encode user input
    user_embedding = model.encode(skills)
    user_embedding = user_embedding.astype(np.float32).reshape(1, -1)
    user_embedding = user_embedding / np.linalg.norm(user_embedding, axis=1, keepdims=True)

    # Perform similarity search
    distances, indices = index.search(user_embedding, k=10)

    # Filter results for diversity
    unique_titles = set()
    diverse_recommendations = []
    for idx in indices.flatten():
        title = df.iloc[idx]["Job Title"]
        if title not in unique_titles:
            diverse_recommendations.append((title, df.iloc[idx]["Cluster"], df.iloc[idx]["skills"]))
            unique_titles.add(title)
        if len(diverse_recommendations) == 5:  # Top 5 diverse results
            break

    # Display results
    print("Diverse Recommendations:")
    for job_title, cluster, required_skills in diverse_recommendations:
        # Convert required skills and user skills to sets
        required_skills_set = set(required_skills.split(", "))
        user_skills_set = set(skills.split(", "))

        # Calculate missing skills
        missing_skills = required_skills_set - user_skills_set

        # Display recommendation and missing skills
        print(f"- {job_title} (Cluster {cluster})")
        # print(f"  Required Skills: {', '.join(required_skills_set)}")
        print(f"  Missing Skills: {', '.join(missing_skills) if missing_skills else 'None'}")


Testing for Role: Data Scientist
Skills: Python, Machine Learning, SQL
Diverse Recommendations:
- Data Analyst (Cluster 0)
  Missing Skills: R), PyTorch) Statistical analysis and modeling Data preprocessing and cleaning Big data technologies (e.g., TensorFlow, Machine learning algorithms and libraries (e.g., Spark) Data visualization Strong programming skills (Python, scikit-learn, Hadoop

Testing for Role: Software Engineer
Skills: Python, Java, Software Development, Algorithms
Diverse Recommendations:
- Software Developer (Cluster 20)
  Missing Skills: React Native, Flutter) Mobile app design principles APIs and web services integration Debugging and troubleshooting, Swift, Kotlin) Cross-platform development (e.g., Mobile app development languages (e.g.

Testing for Role: DevOps Engineer
Skills: AWS, Kubernetes, CI/CD, Docker
Diverse Recommendations:
- Systems Engineer (Cluster 6)
  Missing Skills: Azure) DevOps practices Automation Security in the cloud Disaster recovery Scalabilit